# Autoresearch Vision — Experiment Analysis

Analysis of autonomous architecture search results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated: commit, primary_metric, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["primary_metric"] = pd.to_numeric(df["primary_metric"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()
df["experiment"] = range(len(df))

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep    = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash   = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

kept = df[df["status"] == "KEEP"]
if len(kept) > 0:
    best = kept.loc[kept["primary_metric"].idxmax()]
    print(f"\nBest kept experiment:")
    print(f"  commit:  {best['commit']}")
    print(f"  metric:  {best['primary_metric']:.6f}")
    print(f"  memory:  {best['memory_gb']:.1f} GB")
    print(f"  desc:    {best['description']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Primary metric over experiments ---
ax = axes[0]
color_map = {"KEEP": "#2ecc71", "DISCARD": "#e74c3c", "CRASH": "#95a5a6"}
for status, grp in df.groupby("status"):
    valid = grp[grp["primary_metric"] > 0]
    ax.scatter(valid["experiment"], valid["primary_metric"],
               label=status.title(), color=color_map.get(status, "gray"),
               alpha=0.85, s=60, zorder=3)

# Best-so-far frontier on kept experiments
kept_valid = df[(df["status"] == "KEEP") & (df["primary_metric"] > 0)].sort_values("experiment")
if len(kept_valid) > 0:
    best_so_far = kept_valid["primary_metric"].cummax()
    ax.step(kept_valid["experiment"], best_so_far, where="post",
            color="#2c3e50", linewidth=2, linestyle="--", label="Best so far", zorder=4)

ax.set_xlabel("Experiment #")
ax.set_ylabel("Primary Metric (higher = better)")
ax.set_title("Primary Metric Over Experiments")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: VRAM usage ---
ax = axes[1]
valid_mem = df[df["memory_gb"] > 0]
for status, grp in valid_mem.groupby("status"):
    ax.scatter(grp["experiment"], grp["memory_gb"],
               label=status.title(), color=color_map.get(status, "gray"),
               alpha=0.85, s=60)
ax.axhline(24.0, color="red", linestyle=":", linewidth=1.5, label="4090 limit (24 GB)")
ax.set_xlabel("Experiment #")
ax.set_ylabel("Peak VRAM (GB)")
ax.set_title("VRAM Usage")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 3: Metric vs VRAM scatter (Pareto frontier) ---
ax = axes[2]
valid = df[(df["primary_metric"] > 0) & (df["memory_gb"] > 0)]
for status, grp in valid.groupby("status"):
    ax.scatter(grp["memory_gb"], grp["primary_metric"],
               label=status.title(), color=color_map.get(status, "gray"),
               alpha=0.85, s=60)
ax.set_xlabel("Peak VRAM (GB)")
ax.set_ylabel("Primary Metric")
ax.set_title("Metric vs VRAM (Pareto)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved progress.png")

In [ ]:
# Full results table sorted by metric (best first)
display_cols = ["experiment", "commit", "primary_metric", "memory_gb", "status", "description"]
display_cols = [c for c in display_cols if c in df.columns]
df_sorted = df.sort_values("primary_metric", ascending=False)
print(df_sorted[display_cols].to_string(index=False))